# AudioMoth WAV metadata inspection

This notebook opens a WAV file and extracts:

- basic WAV metadata, such as sample rate and duration
- filename-derived timestamp, where possible
- GUANO metadata, including GPS fields if present


In [1]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path
import re
import wave

import pandas as pd


In [2]:
# Set this to one of your AudioMoth WAV files.
# Example:
# wav_path = Path('../data/raw/example_site/20260416_210000.WAV')

wav_path = Path("../data/raw/20260416_214300.WAV")
wav_path


PosixPath('../data/raw/20260416_214300.WAV')

In [3]:
TIMESTAMP_PATTERNS = (
    re.compile(r'(\d{8}_\d{6})'),
    re.compile(r'(\d{8}T\d{6})'),
)


def extract_timestamp_from_filename(file_name: str) -> datetime | None:
    """Extract a timestamp from a file name if it matches known patterns."""
    for pattern in TIMESTAMP_PATTERNS:
        match = pattern.search(file_name)
        if match is None:
            continue

        candidate = match.group(1)
        for fmt in ('%Y%m%d_%H%M%S', '%Y%m%dT%H%M%S'):
            try:
                return datetime.strptime(candidate, fmt)
            except ValueError:
                continue

    return None


def extract_basic_wav_metadata(file_path: Path) -> dict[str, object]:
    """Read basic metadata from a PCM WAV file."""
    with wave.open(str(file_path), 'rb') as wav_handle:
        frame_rate = wav_handle.getframerate()
        frame_count = wav_handle.getnframes()
        n_channels = wav_handle.getnchannels()
        sample_width_bytes = wav_handle.getsampwidth()
        duration_s = frame_count / frame_rate if frame_rate else None

    return {
        'file_path': str(file_path),
        'file_name': file_path.name,
        'sample_rate_hz': frame_rate,
        'frame_count': frame_count,
        'channels': n_channels,
        'sample_width_bytes': sample_width_bytes,
        'duration_s': duration_s,
        'timestamp_from_filename': extract_timestamp_from_filename(
            file_path.name
        ),
    }


In [4]:
def extract_guano_fields(file_path: Path) -> dict[str, str]:
    """Extract a GUANO block from a WAV file, if present."""
    raw_bytes = file_path.read_bytes()
    marker = b'GUANO|Version'
    start_index = raw_bytes.find(marker)

    if start_index == -1:
        return {}

    guano_text = raw_bytes[start_index:].decode('utf-8', errors='ignore')
    fields: dict[str, str] = {}

    for line in guano_text.splitlines():
        if ':' not in line:
            continue
        key, value = line.split(':', maxsplit=1)
        fields[key.strip()] = value.strip()

    return fields


def extract_guano_location(
    guano_fields: dict[str, str],
) -> tuple[float | None, float | None]:
    """Extract latitude and longitude from the GUANO Loc Position field."""
    loc_position = guano_fields.get('Loc Position')
    
    if loc_position is None:
        return None, None

    parts = loc_position.split(' ')
    if len(parts) != 2:
        return None, None

    try:
        latitude = float(parts[0])
        longitude = float(parts[1])
    except ValueError:
        return None, None

    return latitude, longitude


In [6]:
if not wav_path.exists():
    raise FileNotFoundError(
        f'Update wav_path first. File not found: {wav_path}'
    )

basic_metadata = extract_basic_wav_metadata(wav_path)
guano_fields = extract_guano_fields(wav_path)
latitude, longitude = extract_guano_location(guano_fields)

basic_metadata['guano_present'] = bool(guano_fields)
basic_metadata['guano_latitude'] = latitude
basic_metadata['guano_longitude'] = longitude

pd.DataFrame([basic_metadata]).T.rename(columns={0: 'value'})


,value
file_path,../data/raw/20260416_214300.WAV
file_name,20260416_214300.WAV
sample_rate_hz,250000
frame_count,13750000
channels,1
sample_width_bytes,2
duration_s,55.0
timestamp_from_filename,2026-04-16 21:43:00
guano_present,True
guano_latitude,50.432584


In [7]:
loc_position = guano_fields.get('Loc Position')
loc_position

'50.432584 -3.672039'

In [8]:
if guano_fields:
    display(pd.DataFrame(
        sorted(guano_fields.items()),
        columns=['guano_field', 'value'],
    ))
else:
    print('No GUANO metadata found in this WAV file.')


,guano_field,value
0,Firmware Version,AudioMoth-Firmware-Basic (1.11.1)
1,GUANO|Version,1.0
2,Loc Position,50.432584 -3.672039
3,Make,Open Acoustic Devices
4,Model,AudioMoth
5,OAD|Battery Voltage,3.5
6,OAD|Loc Source,GPS
7,OAD|Recording Settings,250000 GAIN 2
8,Original Filename,20260416_214300.WAV
9,Serial,24F319046907737B


## Notes

- AudioMoth GPS metadata is often stored in GUANO rather than the core WAV header.
- If you do not see coordinates, the file may not contain GUANO GPS fields.
- This notebook is a good first check before turning the logic into pipeline code.
